# 2024 ML Session 6주차 - Bagging & Boosting
- 실습 자료에서는 분류 모델만 다루었으나, 각 모델들은 회귀 문제에도 적용이 가능합니다.
- 버전에 따라 특정 파라미터가 없거나, 변경되거나, 몇몇 옵션을 지원하지 않는 경우가 있습니다.

In [1]:
import warnings
warnings.filterwarnings(action='ignore')

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

## 데이터 불러오기
### DACON 신용카드 사용자 연체 예측 AI 경진대회
원본 데이터에서 일부 column 삭제
- gender: 성별 (1/0)
- car: 차량 소유 여부 (1/0)
- reality: 부동산 소유 여부 (1/0)
- child_num: 자녀 수 (num)
- income_total: 연간 소득 (num)
- income_type: 소득 분류 (cat)
- edu_type: 교육 수준 (cat)
- family_type: 결혼 여부 (cat)
- house_type: 생활 방식 (cat)
- credit : 사용자의 신용카드 대금 연체를 기준으로 한 신용도 (y)

In [2]:
data = pd.read_csv('data/dacon_data.csv')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26457 entries, 0 to 26456
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   gender        26457 non-null  object 
 1   car           26457 non-null  object 
 2   reality       26457 non-null  object 
 3   child_num     26457 non-null  int64  
 4   income_total  26457 non-null  float64
 5   income_type   26457 non-null  object 
 6   edu_type      26457 non-null  object 
 7   family_type   26457 non-null  object 
 8   house_type    26457 non-null  object 
 9   credit        26457 non-null  int64  
dtypes: float64(1), int64(2), object(7)
memory usage: 2.0+ MB


In [6]:
data['credit'].value_counts()

credit
2    16968
1     6267
0     3222
Name: count, dtype: int64

### 데이터 전처리

In [7]:
from sklearn.preprocessing import OrdinalEncoder

In [8]:
x_train, x_test, y_train, y_test = train_test_split(data.iloc[:,:-1], data.iloc[:,-1],
                                                   test_size = 0.2, random_state = 42)

In [9]:
cat_feature = ['gender', 'car', 'reality', 'income_type', 'edu_type', 'family_type', 'house_type']

In [10]:
encoder = OrdinalEncoder()
x_train[cat_feature] = encoder.fit_transform(x_train[cat_feature])
x_test[cat_feature] = encoder.transform(x_test[cat_feature])

In [11]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 21165 entries, 23445 to 23654
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   gender        21165 non-null  float64
 1   car           21165 non-null  float64
 2   reality       21165 non-null  float64
 3   child_num     21165 non-null  int64  
 4   income_total  21165 non-null  float64
 5   income_type   21165 non-null  float64
 6   edu_type      21165 non-null  float64
 7   family_type   21165 non-null  float64
 8   house_type    21165 non-null  float64
dtypes: float64(8), int64(1)
memory usage: 1.6 MB


# Bagging
- 학습 데이터에서 복원 추출(bootstrap)하여 서브셋 생성 후 해당 서브셋을 별개의 모델에 학습시킨 뒤 각 모델의 예측 결과를 종합하는 앙상블 기법 

## BaggingClassifier
- sklearn에서 지원하는 기본 Bagging 분류 모델
### parameter
- base_estimator : bagging에 사용될 예측기 (default = DecisionTreeClassifier)
- n_estimator : base_estimator의 갯수 (default = 10)
- bootstrap : bootstrap 여부 (default = True)
- oob_score : 모델 평가에 oob 샘플의 사용 여부 (default = False)

In [12]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

In [13]:
bg = BaggingClassifier(base_estimator = DecisionTreeClassifier())
bg.fit(x_train, y_train)
pred = bg.predict(x_test)
print("BaggingClassifier :", f1_score(y_test, pred, average='micro'))

# macro : 각 클래스별로 성능 지표 계산 후 평균 내어 최종 값 산출
# micro : 모든 클래스를 합산한 다음 합산된 값으로 성능 지표 계산

BaggingClassifier : 0.6466364323507181


## RandomForestClassifier
- sklearn에서 지원하는 RandomForest 분류 모델
- <span style="color:gray">(version 1.1 이전)</span> DecisionTree와 같이 모든 feature의 information gain을 고려하여 노드를 분할
- <span style="color:gray">(version 1.1 이후)</span> 선택된 N개의 feature를 랜덤으로 $\sqrt(N)$개로 나눈 후 그 중 informatin gain을 고려하여 노드를 분할
### parameter
- **n_estimator** : 트리의 수
- criterion : 분할 여부 계산하는 함수 (gini / entropy)
- **max_depth** : 트리의 최대 깊이 (None - 트리가 완성될 때까지 계속 노드 확장)
- **min_samples_split** : 노드 분할하기 위한 최소 샘플 수
- **min_samples_leaf** : 리프 노드에 있어야 할 최소 샘플 수
- min_weight_fraction_leaf : 리프 노드에 있어야 할 샘플의 최소 가중치 합
- **max_features** : 최적 분할을 찾을 때 고려할 피처의 수 (auto / sprt / log2 / 정수)
- **max_leaf_nodes** : 리프 노드의 최대 수 (None - 제한 없음)
- min_impurity_decrease : 노드 분할을 위한 불순도 감소 최소량 (=노드 분할의 기준)
- bootstrap : 부트스트랩 사용할지 여부
- oob_score : out-of-bag 샘플로 정확도 추정할 것인지 여부
- n_jobs : 병렬 작업 수 (-1이면 모든 프로세서 사용)
- random_state : random 시드
- verbose : 트리 구축 과정 텍스트 출력 여부 (1이면 출력)
- warm_start : 이전 호출 재사용 여부
- class_weight : 데이터셋 클래스 불균형 조절 위한 클래스 가중치 (balanced:빈도 낮은 클래스에 높은 가중치/balanced_subsample:서브셋 기준으로 가중치)
- ccp_alpha : 트리 복잡도 페널티 값
- max_samples : 학습 데이터셋에서 샘플링 되는 샘플의 수


**아래 코드는 각 parameter의 default 값 (버전에 따라 약간씩 차이 존재)**

In [14]:
from sklearn.ensemble import RandomForestClassifier

In [15]:
rf = RandomForestClassifier(
    n_estimators=100,
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_features='sqrt',
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    oob_score=False,
    n_jobs=None,
    random_state=None,
    verbose=0,
    warm_start=False,
    class_weight=None,
    ccp_alpha=0.0,
    max_samples=None
)

In [16]:
rf.fit(x_train, y_train)
pred = rf.predict(x_test)
print("RandomForestClassifier :", f1_score(y_test, pred, average='micro'))

RandomForestClassifier : 0.6511715797430083


# Boosting
- 약한 학습기를 오류를 개선하는 방향으로 연속적/순차적으로 학습시키는 앙상블 기법
## AdaBoostClassifier
- sklearn에서 지원하는 AdaBoost 분류 모델
### parameter
- **base_estimator** : 부스팅에 사용할 기본 모델
- **n_estimator** : 사용할 예측기 수 (부스팅 단계 수)
- **learning_rate**: 부스팅에 사용할 학습률
- algorithm : adaboost 구현 알고리즘 (SAMME: 기본 다중클래스 / SAMME.R: 개선된 버전)
- random_state : random 시드

**아래 코드는 각 parameter의 default 값 (버전에 따라 약간씩 차이 존재)**

In [17]:
from sklearn.ensemble import AdaBoostClassifier

In [18]:
ada = AdaBoostClassifier(
    base_estimator= DecisionTreeClassifier(),
    n_estimators=50,
    learning_rate=1.0,
    algorithm='SAMME.R',
    random_state=None
)

In [19]:
ada.fit(x_train, y_train)
pred = ada.predict(x_test)
print('AdaBoost : ', f1_score(y_test, pred, average = 'micro'))

AdaBoost :  0.6490929705215419


## GradientBoostingClassifier
- sklearn에서 지원하는 GBM 분류 모델

### parameter
- loss : 최적화할 손실 함수 (log_loss: 이진&다중분류 / exponenetial : 이진분류)
- **learning_rate** : 학습률
- **n_estimators** : 사용할 예측기 수 (부스팅 단계 수)
- subsample : 각 부스팅 단계에서 사용할 훈련 샘플의 비율
- criterion : 분할 여부 결정하는 함수 (friedman_mse/mse/mae)
- **min_samples_split** : 노드 분할에 필요한 최소 샘플 수
- **min_samples_leaf** : 리프 노드에 필요한 최소 샘플 수
- min_weight_fraction_leaf : 리프 노드에 필요한 샘플의 최소 가중치
- **max_depth** : 개별 트리의 최대 깊이 지정
- min_impurity_decrese : 노드를 분할하기 위해 필요한 불순도의 최소 감소량
- init : 초기값 (None - 초기값으로 평균 사용)
- random_state : random 시드
- max_feature : 각 노드에서 사용할 피처 수
- verbose : 학습 시 메시지 출력 여부 (1: 반복마다 출력 / 2: 트리 추가할 때마다 출력)
- max_leaf_nodes : 각 트리에서 가질 수 있는 최대 리프 노드 수
- warm_start : 이전 호출의 학습 내용 재사용 여부
- validation_fraction : 조기 종료를 위해 예약된 훈련 데이터의 비율
- **n_iter_no_change** : n번 동안 검증 점수가 향상되지 않으면 학습 중단하겠다 할 때의 그 n번 지정
- tol : 조기 종료를 위한 허용 오차 (tol보다 작게 향상되면 향상되지 않았다고 간주)
- ccp_alpha : 모델 복잡도 줄이기 위한 가지치기 매개변수 (페널티값. 줄일수록 트리 간소화)

**아래 코드는 각 parameter의 default 값 (버전에 따라 약간씩 차이 존재)**

In [20]:
from sklearn.ensemble import GradientBoostingClassifier

In [21]:
gbm = GradientBoostingClassifier(
    loss='log_loss',
    learning_rate=0.1,
    n_estimators=100,
    subsample=1.0,
    criterion='friedman_mse',
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_depth=3,
    min_impurity_decrease=0.0,
    init=None,
    random_state=None,
    max_features=None,
    verbose=0,
    max_leaf_nodes=None,
    warm_start=False,
    validation_fraction=0.1,
    n_iter_no_change=None,
    tol=0.0001,
    ccp_alpha=0.0
)

In [22]:
gbm.fit(x_train, y_train)
pred = gbm.predict(x_test)
print('GBM : ', f1_score(y_test, pred, average='micro'))

GBM :  0.6385109599395313


## XGBoost
- xgboost 공식 라이브러리
- 설치가 안 되어 있을 경우 ``!pip install xgboost``로 설치 필요
### parameter
- **booster** : 부스팅에 사용할 알고리즘 유형 (gbtree / gblinear / dart)
- silent : 학습 시 메시지 출력 여부 (0: 안뜸 / 1: 경고만 / 2: 일반적 정보 / 3: 모든 정보)

- **learning rate** : 학습률
- **min_child_weight** : 자식 노드에서 필요한 모든 관측치에 대한 가중치의 최소 합 (클수록 보수적 = 과적합 적어짐)
- **max_depth** : 개별 트리의 최대 깊이 (클수록 과적합 가능성 ↑)
- gamma : 리프 노드 추가로 분할하기 위한 최소 손실 감소량
- **subsample** : 학습 데이터의 샘플링 비율 (보통 0.5~1 사이)
- colsample_bytree : 트리 구성시 열의 샘플링 비율 (낮을수록 과적합 ↓)
- colsample_bylevel : 트리 레벨별 열의 서브 샘플링 비율
- colsample_bynode : 트리 노드별 열의 서브 샘플링 비율
- **reg_lambda** : L2 정규화 가중치 (클수록 과적합 ↓)
- **reg_alpha** : L1 정규화 가중치 (클수록 과적합 ↓)
- tree_method : 트리 구성 알고리즘 (auto / exact / approx / hist / gpu_hist)
- precess_type : 업데이터 선택 기준 (defalut / update)
- grow_policy : 트리 생성 전략 (depthwise / lossguide)

- objective : 손실 함수 (reg:squarederror / binary:logistic / multi:softmax)
- base_score : 초기 예측 점수
- eval_metric : 검증 데이터에 사용되는 평가 지표
- seed : random seed

- n_estimators : 부스팅 단계 수

**아래 코드는 각 parameter의 default 값 (버전에 따라 약간씩 차이 존재)**

In [23]:
from xgboost import XGBClassifier

In [24]:
xgb = XGBClassifier(
    n_estimators=100,
    booster="gbtree",
    verbosity=1,
    learning_rate=0.3,
    min_child_weight=1,
    max_depth=6,
    max_leaf_nodes=None,
    gamma=0,
    subsample=1,
    colsample_bytree=1,
    colsample_bylevel=1,
    colsample_bynode=1,
    reg_lambda= 1,
    reg_alpha=0,
    tree_method="auto",
    grow_policy="depthwise",
    objective="multi:softmax",
    base_score=0.5,
    eval_metric="logloss",
    seed=0
)

In [25]:
xgb.fit(x_train, y_train)
pred = xgb.predict(x_test)
print('XGBoost : ', f1_score(y_test, pred, average='micro'))

XGBoost :  0.645124716553288


## LGBM
- lightgbm 공식 라이브러리
- 설치가 안 되어 있을 경우 `!pip install lightgbm`로 설치 필요
### parameter
- **boosting_type** : 부스팅 알고리즘 (dart, goss, rf)
- **num_leaves** : 하나의 트리가 가질 수 있는 최대 잎의 수. (너무 크면 과적합)
- **max_depth** : 트리의 최대 깊이 (-1은 제한 없음. 너무 크면 과적합)
- **learning_rate** : 학습률
- **n_estimators** : 부스팅 단계 수
- objective : 목적함수 (task에 따라 종류 매우 多)
- class_weight : 클래스 가중치
- min_split_gain : 트리 노드를 분할하는 데 필요한 최소 손실 감소량
- min_child_weight : 리프 노드에 필요한 데이터의 최소 가중치 합 (과적합 방지)
- **min_child_samples** : 리프 노드에서 필요한 최소 샘플 수 (과적합 방지)
- **subsample** : 훈련 데이터의 sampling 비율
- subsample_freq : 서브 샘플링의 빈도 (0 : 사용되지 않음)
- reg_alpha : L1 정규화 항
- reg_lambda : L2 정규화 항
- random_state : random 시드
- n_jobs : 병렬 코어 수 (-1 : 전부 사용)
- importance_type : 중요도 계산 기준 (split: 분할 횟수 / gain: 분할로 얻는 이득)
- max_bin : 피처 값을 나눌 최대 bin 수
- min_data_in_bin : 각 bin에 필요한 데이터의 최소 수
- bin_construct_sample_cnt : bin 구성에 사용되는 샘플 수

paramter 공식 문서 : https://lightgbm.readthedocs.io/en/stable/Parameters.html#core-parameters

**아래 코드는 각 parameter의 default 값 (버전에 따라 약간씩 차이 존재)**

In [26]:
from lightgbm import LGBMClassifier

In [27]:
lgbm = LGBMClassifier(
    boosting_type='gbdt',
    num_leaves=31,
    max_depth=-1,
    learning_rate=0.1,
    n_estimators=100,
    objective=None,
    class_weight=None,
    min_split_gain=0.0,
    min_child_weight=0.001,
    min_child_samples=20,
    subsample=1.0,
    subsample_freq=0,
    reg_alpha=0.0,
    reg_lambda=0.0,
    random_state=None,
    n_jobs=-1,
    importance_type='split',
    max_bin=255,
    min_data_in_bin=3,
    bin_construct_sample_cnt=200000,
)

In [28]:
lgbm.fit(x_train, y_train)
pred = lgbm.predict(x_test)
print('LGBM : ', f1_score(y_test, pred, average='micro'))

[LightGBM] [Warning] bin_construct_sample_cnt is set=200000, subsample_for_bin=200000 will be ignored. Current value: bin_construct_sample_cnt=200000
[LightGBM] [Warning] bin_construct_sample_cnt is set=200000, subsample_for_bin=200000 will be ignored. Current value: bin_construct_sample_cnt=200000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002950 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 213
[LightGBM] [Info] Number of data points in the train set: 21165, number of used features: 9
[LightGBM] [Info] Start training from score -2.107665
[LightGBM] [Info] Start training from score -1.441911
[LightGBM] [Info] Start training from score -0.443162
[LightGBM] [Warning] bin_construct_sample_cnt is set=200000, subsample_for_bin=200000 will be ignored. Current value: bin_construct_sample_cnt=200000
LGBM :  0.6443688586545729


## CatBoost
- catboost 공식 라이브러리
- 설치가 안 되어 있을 경우 `!pip install catboost`로 설치 필요
### 주요 parameter
**이외에도 많은 파라미터가 있지만, 50개가 넘어가서 생략함.**
- iteration : 부스팅 단계 수
- learning_rate : 학습률
- max_depth : 트리의 최대 깊이
- l2_leaf_reg : L2 정규화 항 (과적합 방지)
- rsm : 각 트리에서 사용할 feature의 비율
- loss_function : 최적화 손실 함수
- border_count : 수치형 feature를 몇 개의 묶음으로 나눌 것인지 (많을수록 정확하지만 속도 ↓)
- od_pval : 과적합 감지를 위한 p-value 임계값 (값이 낮을수록 과적합에 민감해짐)
- od_wait : 과적합 감지 후 추가로 반복할 횟수
- od_type : 과적합 감지 유형 (incToDec: 손실이 감소하다가 다시 증가할 때/ Iter: od_wait 동안 향상이 없을 때)
- nan_mode : 결측치(NaN) 어떻게 처리할지 결정
- leaf_estimation_method : 리프의 값을 추정하는 방법
- thread_count : 학습에 사용할 코어 수
- random_seed : random 시드
- use_best_model : 검증 데이터(test)에서 최적의 모델을 사용할지 여부
- verbose : 메시지 출력 여부
- ctr_leaf_count_limit : 범주형 feature의 범주 수가 이 값보다 크면 내부 계산 제한
- one_hot_max_size : 이 숫자 이하의 클래스 개수를 가지는 변수는 one-hot 처리
- bootstrap_type : 부트스트랩 유형 (Bayesian: 각 반복마다 샘플링 확률 업데이트 / Bernoulli: 각 객체 독립적 샘플링 ~ RF / MVS: 최소 분산 샘플링 / No : 부트스트랩 안함)
- subsample : 부트스트랩 샘플 비율
- grow_policy : 트리 성장 정책 (SymmetricTree / Depthwise / Lossguide)
- min_child_samples : 리프에 필요한 최소 데이터 수 (너무 작으면 과적합)
- boost_from_average : 초기 예측을 평균에서 시작할지 여부

In [30]:
pip install catboost

  Obtaining dependency information for catboost from https://files.pythonhosted.org/packages/1c/e1/78e635a1e5f0066bd02a1ecfd658ad09fe30d275c65c2d0dd76fe253e648/catboost-1.2.7-cp311-cp311-win_amd64.whl.metadata
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB 131.3 kB/s eta 0:12:55
   ---------------------------------------- 0.0/101.7 MB 131.3 kB/s eta 0:12:55
   ---------------------------------------- 0.0/101.7 MB 178.6 kB/s eta 0:09:30
   ---------------------------------------- 0.0/101.7 MB 178.6 kB/s eta 0:09:30
   ---------------------------------------- 0.1/101.7 MB 286.7 kB/s eta 0:05:55
   ---------------------------------------- 0.1/101.7 MB 379.3 kB/s eta 0:04:28
   ---------------------------------------- 0.2/101.7 MB 656.0 kB/s eta 0:02:35
   --------------

In [31]:
from catboost import CatBoostClassifier

In [32]:
x_train, x_test, y_train, y_test = train_test_split(data.iloc[:,:-1], data.iloc[:,-1],
                                                   test_size = 0.2, random_state = 42)

In [33]:
cat_feature = ['gender', 'car', 'reality', 'income_type', 'edu_type', 'family_type', 'house_type']

In [34]:
catb = CatBoostClassifier(
    iterations=1000,
    learning_rate=None,
    depth=6,
    l2_leaf_reg=3.0,
    rsm=1.0,
    loss_function='MultiClass',
    border_count=254,
    od_pval=None,
    od_wait=20,
    od_type='IncToDec',
    nan_mode='Min',
    leaf_estimation_method='Newton',
    thread_count=None,
    random_seed=None,
    use_best_model=False,
    ctr_leaf_count_limit=None,
    one_hot_max_size=2,
    bootstrap_type='Bayesian',
    subsample=None,
    grow_policy='SymmetricTree',
    min_data_in_leaf=1,
    boost_from_average=False,
    verbose = False,
    
    # 범주형 변수 리스트 형태로 넣어주면 알아서 처리해줌
    cat_features = cat_feature
)

In [35]:
catb.fit(x_train, y_train)
pred = catb.predict(x_test)
print('CatBoost', f1_score(y_test, pred, average='micro'))

CatBoost 0.6405895691609977
